# EDA — Clasificación comercial de limón (5 vistas por fruto)

**Objetivo:** conocer los datos *antes* de modelar: ¿están completos, balanceados, bien etiquetados, y libres de atajos (lote/fecha/iluminación) que inflen las métricas?

**Antes de ejecutar:** corre `python src/ingesta.py` (datos reales) o `python scripts/generar_datos_demo.py` + `python src/ingesta.py --raw data/demo/raw --salida data/demo/processed` (datos sintéticos de prueba).

> Con `MODO_DEMO = True` los resultados son de imágenes sintéticas: sirven para validar el notebook, **no** para sacar conclusiones.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cv2
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import eda
from src.regla_comercial import CALIBRE_EXTRA, CALIBRE_PRIMERA, CALIBRE_SEGUNDA

MODO_DEMO = True          # <-- poner False con los datos reales
PROC = ROOT / "data" / ("demo/processed" if MODO_DEMO else "processed")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .25})
PAL = {"Extra": "#2e7d32", "Primera": "#7cb342", "Segunda": "#fbc02d", "Tercera": "#ef6c00", "Fabrica": "#7a1f2b"}

## 1. Carga de índices

In [ ]:
frutos = pd.read_csv(PROC / "indice_frutos.csv", dtype={"id_fruto": str, "lote": str, "fecha": str, "vistas_faltantes": str})
imgs   = pd.read_csv(PROC / "indice_imagenes.csv", dtype={"id_fruto": str, "lote": str, "fecha": str})
frutos["cat"] = eda.normalizar_categorias(frutos["categoria_ref"])
print(f"Frutos: {len(frutos)} | Imágenes: {len(imgs)} | Esperadas (5 vistas): {len(frutos)*5}")
frutos.head()

## 2. Calidad de datos y etiquetas
Qué buscar: duplicados, etiquetas vacías o inválidas, calibres imposibles, frutos sin imágenes (o imágenes sin etiqueta), capturas incompletas.

In [ ]:
display(eda.resumen_calidad_etiquetas(frutos))
inc = frutos[~frutos["completo"]][["id_fruto", "lote", "fecha", "n_vistas", "vistas_faltantes", "origen"]]
print("\nFrutos con captura incompleta (protocolo: falta 1 vista ecuatorial → 'captura parcial'; falta pedúnculo o >1 ecuatorial → excluir y registrar):")
display(inc)

## 3. Balance de clases vs. cuotas mínimas
Qué buscar: categorías que no llegan a su cuota (250/250/250/200/200). Las minoritarias (Tercera, Fábrica) suelen ser las que faltan.

In [ ]:
cuotas = eda.tabla_cuotas(frutos); display(cuotas)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(cuotas.index, cuotas["frutos"], color=[PAL[c] for c in cuotas.index], label="frutos actuales")
ax.plot(cuotas.index, cuotas["cuota_min"], "k_", ms=28, mew=2, label="cuota mínima")
ax.set_ylabel("frutos"); ax.set_title("Frutos por categoría vs. cuota"); ax.legend(); plt.show()

## 4. Cobertura de lotes y fechas
Qué buscar: ≥3 lotes y ≥4 fechas, y que ninguna categoría quede concentrada en un solo lote/fecha.

In [ ]:
cob = eda.cobertura_lote_fecha(frutos)
print(f"Lotes: {cob['n_lotes']} (mín. 3 → {'OK' if cob['cumple_lotes'] else 'NO CUMPLE'}) | "
      f"Fechas: {cob['n_fechas']} (mín. 4 → {'OK' if cob['cumple_fechas'] else 'NO CUMPLE'})")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, key, tit in [(axs[0], "cat_x_lote", "Categoría × lote"), (axs[1], "cat_x_fecha", "Categoría × fecha")]:
    t = cob[key]; im = ax.imshow(t.values, cmap="YlOrBr", aspect="auto")
    ax.set_xticks(range(t.shape[1])); ax.set_xticklabels(t.columns, rotation=45); ax.set_yticks(range(t.shape[0])); ax.set_yticklabels(t.index)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]): ax.text(j, i, t.values[i, j], ha="center", va="center", fontsize=9)
    ax.set_title(tit); ax.grid(False)
plt.tight_layout(); plt.show()

## 5. Calibre: distribución y consistencia con la categoría
Qué buscar: calibres por categoría coherentes con los umbrales (35 / 37 / 39 mm), etiquetas que contradicen el calibre (Fábrica se excluye: la decide defecto/color) y frutos *fronterizos* (±0,5 mm de un umbral), que son los más sensibles al error de medición (meta MAE ≤ 1,0 mm).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].hist(frutos["calibre_mm_ref"].dropna(), bins=25, color="#7a1f2b", alpha=.8)
for u in (CALIBRE_SEGUNDA, CALIBRE_PRIMERA, CALIBRE_EXTRA): axs[0].axvline(u, color="k", ls="--", lw=1)
axs[0].set_title("Calibre de referencia (mm) — líneas: umbrales"); axs[0].set_xlabel("mm")
data = [frutos.loc[frutos.cat == c, "calibre_mm_ref"].dropna() for c in eda.CATEGORIAS]
bp = axs[1].boxplot(data, labels=eda.CATEGORIAS, patch_artist=True)
for patch, c in zip(bp["boxes"], eda.CATEGORIAS): patch.set_facecolor(PAL[c]); patch.set_alpha(.7)
for u in (CALIBRE_SEGUNDA, CALIBRE_PRIMERA, CALIBRE_EXTRA): axs[1].axhline(u, color="k", ls="--", lw=1)
axs[1].set_title("Calibre por categoría"); axs[1].set_ylabel("mm"); plt.tight_layout(); plt.show()

inco = eda.inconsistencias_calibre_categoria(frutos)
print(f"Etiquetas no-Fábrica que contradicen su calibre: {len(inco)} (revisar con el panel / recalibrar medición)")
display(inco)
fr = eda.fronterizos(frutos, 0.5)
print(f"Frutos fronterizos (±0,5 mm de un umbral): {len(fr)} = {100*len(fr)/len(frutos):.1f}% del total")

## 6. Color y defecto (panel de agrónomos)
Qué buscar: que Fábrica se explique por defecto alto **o** color amarillo, y que las demás categorías tengan defecto bajo. Si estas columnas no existen en tus etiquetas, se omite.

In [ ]:
if {"color_panel", "severidad_defecto_panel"} <= set(frutos.columns):
    display(pd.crosstab(frutos["cat"], frutos["color_panel"]).reindex(eda.CATEGORIAS))
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.boxplot([frutos.loc[frutos.cat == c, "severidad_defecto_panel"].dropna() for c in eda.CATEGORIAS],
               labels=eda.CATEGORIAS)
    ax.set_title("Severidad de defecto por categoría"); plt.show()
else:
    print("Sin columnas de panel de color/defecto: omitido.")

## 7. Calidad de las imágenes
Qué buscar: resolución uniforme, imágenes borrosas (nitidez baja), sobre/sub-exposición, **duplicados exactos** y archivos ilegibles. Con muchas imágenes, usar `max_imgs` para una muestra.

In [ ]:
met = eda.calcular_metricas_imagenes(imgs, max_imgs=None)
print("Ilegibles:", int((~met.legible).sum()), "| Duplicadas:", int(met.duplicada.sum()))
print("Resoluciones:", met.groupby(["ancho", "alto"]).size().to_dict())
met["borrosa"] = eda.marcar_atipicas(met, "nitidez") & (met["nitidez"] < met["nitidez"].median())
fig, axs = plt.subplots(1, 3, figsize=(12, 3.3))
for ax, col in zip(axs, ["nitidez", "brillo", "saturacion"]):
    ax.hist(met[col].dropna(), bins=30, color="#555"); ax.set_title(col)
plt.tight_layout(); plt.show()
print("Imágenes muy borrosas (candidatas a recapturar):")
display(met[met.borrosa][["id_fruto", "vista", "nitidez"]])
print("Imágenes duplicadas exactas:")
display(met[met.duplicada][["id_fruto", "vista", "ruta"]])

In [ ]:
# Muestra visual: las 5 vistas de un fruto por categoría
fig, axs = plt.subplots(5, 5, figsize=(13, 10))
for i, cat in enumerate(eda.CATEGORIAS):
    f = frutos[(frutos.cat == cat) & frutos.completo].head(1)
    if f.empty: [a.axis("off") for a in axs[i]]; continue
    idf = f.iloc[0]["id_fruto"]
    for j, v in enumerate(eda.__dict__.get("VISTAS", ["0", "90", "180", "270", "pedunculo"])):
        ax = axs[i, j]; ax.axis("off")
        r = imgs[(imgs.id_fruto == idf) & (imgs.vista == v)]
        if len(r):
            p = Path(r.iloc[0]["ruta"]); p = p if p.is_absolute() else ROOT / p
            ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{cat} · {idf} · {v}", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Riesgo de atajos (leakage de contexto)
Si la categoría se puede predecir por **lote/fecha** o por **brillo de la sesión**, el modelo podría aprender el contexto y no el fruto. Qué buscar: V de Cramér cercano a 1 y brillo/nitidez muy distintos entre lotes → controlar iluminación (carta de color) y no usar lote/fecha como entrada.

In [ ]:
for col in ["lote", "fecha"]:
    print(f"V de Cramér categoría ~ {col}: {eda.asociacion_categoria_metadato(frutos, col):.2f}  (0 = independiente, 1 = totalmente asociado)")
m2 = met.merge(frutos[["id_fruto", "cat"]], on="id_fruto", how="left")
print("\nBrillo y nitidez medios por lote (diferencias grandes = iluminación distinta entre sesiones):")
display(m2.groupby("lote")[["brillo", "nitidez", "saturacion"]].mean().round(1))
fig, axs = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (by, col) in zip(axs, [("lote", "brillo"), ("fecha", "brillo"), ("lote", "nitidez")]):
    grupos = sorted(m2[by].dropna().unique())
    ax.boxplot([m2.loc[m2[by] == g, col] for g in grupos], labels=grupos)
    ax.set_title(f"{col} por {by}"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 9. Vista previa de la partición 70/15/15
**No es la partición oficial** (esa se genera una sola vez y se congela). Sirve para verificar que, al partir por fruto y estratificar por categoría, cada subconjunto conserve todas las categorías, lotes y fechas.

In [ ]:
part = eda.vista_previa_particion(frutos, semilla=0)
display(pd.crosstab(part["cat"], part["subset"]).reindex(eda.CATEGORIAS)[["train", "val", "test"]])
for col in ["lote", "fecha"]:
    t = pd.crosstab(part[col], part["subset"])[["train", "val", "test"]]
    print(f"\n{col}: " + ("OK, todos presentes en los 3 subconjuntos" if (t > 0).all().all() else "⚠ hay valores ausentes en algún subconjunto"))
    display(t)

## 10. Hallazgos y decisiones (completar tras correr con datos reales)

| # | Hallazgo | Evidencia (celda) | Decisión / acción |
|---|---|---|---|
| 1 | ¿Se cumplen las cuotas por categoría? | §3 | |
| 2 | ¿Hay ≥3 lotes y ≥4 fechas, sin concentración por categoría? | §4 | |
| 3 | ¿Etiquetas inconsistentes con el calibre? ¿Cuántas y por qué? | §5 | |
| 4 | ¿% de frutos fronterizos? ¿Impacto en el MAE objetivo? | §5 | |
| 5 | ¿Capturas incompletas, borrosas o duplicadas? | §2, §7 | |
| 6 | ¿Riesgo de atajo por lote/fecha/brillo? | §8 | |
| 7 | ¿La partición 70/15/15 por fruto conserva todas las categorías/lotes/fechas? | §9 | |